In [1]:
from src.utils import get_data_env
from src.models import SpatialGNNModel
from src.dataloading import DeepDataLoader

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
disease_name    = 'influenza'
model           = 'tgcn'
nuts_level      = 'nuts3'
min_date        ='2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False


horizon_size    = 3
horizon_leadtime= 3
sequence_length = 1
lags            = 4

graphtype       = 'boolean_neighbors_self'
modelname       = 'boolean neighbors - spatial gcn'

# training hparams
n_epochs        = 100
lr              = 0.0005
min_delta       = 0.0001
loss            = 'exp_decay'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :   {'mode': 'min', 'factor': 0.5, 'patience': 7},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }


In [ ]:

epidata_loader_bn = DeepDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
epidata_loader_bn.add_time_features()
epidata_loader_bn.log_transform_target()
epidata_loader_bn.set_splits(split_trainval, split_valtest)
epidata_loader_bn.normalize()
epidata_loader_bn.add_lagged_features(lags = lags)
epidata_loader_bn.finalize()
epidata_loader_bn.retrieve_graph(graphtype)
epidata_loader_bn.construct_dataloaders()

spatial_bn = SpatialGNNModel(epidata_loader_bn, name = modelname)
spatial_bn.set_model_hparams()
spatial_bn.set_global_hparams(**global_hparams)
spatial_bn.train(verbose = 2, dataloader_snapshot=False, show_loss= True)
spatial_bn.forecast('test')
spatial_bn.show_forecasts('test', [26,391, 69], target_h=0)
spatial_bn.show_forecasts('test', [26,391, 69], target_h=1)
spatial_bn.show_forecasts('test', [26,391, 69], target_h=2)

Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-04-03 (+6 weeks)
